# Initial Setup

---
**Homework version (`rome_submission.ipynb`).** All original cells are kept in order. Changes are marked `[HW PATCH]`: Covered is installed next to iverilog, the model is gpt-4o through the OpenAI backend, the API key is read from a Colab secret, and the PPA cell uses the selected backend. Parts I-IV of the assignment follow the original cells.

In [ ]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog verilator gtkwave covered   # [HW PATCH] + covered (coverage_exp flow)

In [ ]:
#@title Setting up PPA estimation

%%bash
set -e
cd /content

URL=$(python3 - <<'PY'
import json, urllib.request
data = json.load(urllib.request.urlopen(
    "https://api.github.com/repos/YosysHQ/oss-cad-suite-build/releases/latest"
))
asset = next(
    a for a in data["assets"]
    if "linux-x64" in a["name"] and a["name"].endswith(".tgz")
)
print(asset["browser_download_url"])
PY
)

wget -q -O oss-cad-suite.tgz "$URL"
rm -rf /content/oss-cad-suite /content/oss-cad-suite-install
mkdir -p /content/oss-cad-suite-install
tar -xzf oss-cad-suite.tgz -C /content/oss-cad-suite-install
mv /content/oss-cad-suite-install/oss-cad-suite /content/oss-cad-suite

/content/oss-cad-suite/bin/yosys -V



In [ ]:
#@title Select Model
#define the model to be used
#model_choice = "gpt-5.2"
#model_choice = "gpt-5.2-codex"
model_choice = "gpt-4o"   # [HW PATCH] own OpenAI key
#model_choice = "claude-sonnet-4-6"
#model_choice = "gemini-2.5-flash-preview-04-17"
#model_choice = "gemini-2.5-flash"
#model_choice = "nyu-dice-lab/VeriThoughts-Reasoning-14B" #path to your desired Hugging Face model

In [ ]:
#@title Utility functions

import sys
import os
import time
import re
from abc import ABC, abstractmethod

import openai
import anthropic

try:
    import google.genai.errors
    from google import genai
    from google.genai import types
except Exception:
    genai = None
    types = None

################################################################################
### LOGGING
################################################################################

class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        return self.messages

    def get_last_n_messages(self, n):
        return self.messages[-n:]

    def remove_message(self, index):
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        self.messages = []

    def __str__(self):
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])


################################################################################
### LLM CLASSES
################################################################################

class AbstractLLM(ABC):
    @abstractmethod
    def generate(self, conversation: Conversation):
        pass


class ChatGPT(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        openai.api_key = os.environ["OPENAI_API_KEY"]
        self.client = openai.OpenAI()
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [
            {
                "role": msg["role"] if msg["role"] in ["system", "user", "assistant"] else "user",
                "content": msg["content"]
            }
            for msg in conversation.get_messages()
        ]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=messages,
        )

        return response.choices[0].message.content


class Claude(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ["CLAUDE_API_KEY"])
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20

        messages = [
            {
                "role": msg["role"] if msg["role"] in ["user", "assistant"] else "user",
                "content": msg["content"]
            }
            for msg in conversation.get_messages()
        ]

        for attempt in range(1, max_retries + 1):
            try:
                output = self.client.messages.create(
                    model=self.model_id,
                    max_tokens=16384,
                    messages=messages
                ).content[0].text
                return output

            except Exception as e:
                wait_time = base_delay * (2 ** (attempt - 1))
                print(f"[Retry {attempt}/{max_retries}] Claude API error: {e}. Retrying in {wait_time:.1f} seconds...")
                time.sleep(wait_time)

        print(f"Failed, exceeded max retries {max_retries}")
        return ""


class Gemini(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        if genai is None:
            raise ImportError("google-genai is not installed or could not be imported.")
        self.gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        output = self.gemini_client.models.generate_content(
            model=self.model_id,
            contents=[msg["content"] for msg in conversation.get_messages()],
            config=types.GenerateContentConfig(
                max_output_tokens=16384,
                temperature=0.6,
                topP=0.95,
            )
        ).text

        return output


class HuggingFace(AbstractLLM):
    """
    Local Hugging Face model backend.

    Downloads the selected model from the Hugging Face Hub using snapshot_download,
    then loads it with transformers. The model is cached and reused so it is not
    reloaded every time generate_verilog() is called.
    """

    _loaded_model_id = None
    _tokenizer = None
    _model = None
    _local_model_path = None

    def __init__(
        self,
        model_id=None,
        max_new_tokens=4096,
        temperature=0.2,
        top_p=0.95,
        load_in_4bit=False,
        trust_remote_code=True,
        strip_thinking=True,
    ):
        super().__init__()

        self.model_id = model_id or model_choice

        # If the user accidentally wrote "/org/model", convert it to "org/model".
        # Absolute local paths like "/content/my_model" are still allowed.
        if self.model_id.startswith("/") and not self.model_id.startswith("/content/"):
            self.model_id = self.model_id[1:]

        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p
        self.load_in_4bit = load_in_4bit
        self.trust_remote_code = trust_remote_code
        self.strip_thinking = strip_thinking

        self.hf_token = (
            os.environ.get("HF_TOKEN")
            or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
            or None
        )

        self.cache_dir = os.environ.get("HF_HOME", "/content/huggingface")

        self._ensure_model_loaded()

    def _get_torch_dtype(self):
        import torch

        if not torch.cuda.is_available():
            return torch.float32

        major, minor = torch.cuda.get_device_capability(0)

        # A100/H100 and newer generally support bf16 well.
        if major >= 8:
            return torch.bfloat16

        # T4/V100 are safer with fp16.
        return torch.float16

    def _ensure_model_loaded(self):
        if HuggingFace._model is not None and HuggingFace._loaded_model_id == self.model_id:
            return

        import torch
        from huggingface_hub import snapshot_download
        from transformers import AutoTokenizer, AutoModelForCausalLM

        print(f"[HuggingFace] Downloading/caching model: {self.model_id}")

        # If this is a local path, do not call snapshot_download.
        if os.path.isdir(self.model_id):
            local_model_path = self.model_id
        else:
            local_model_path = snapshot_download(
                repo_id=self.model_id,
                cache_dir=self.cache_dir,
                token=self.hf_token,
            )

        print(f"[HuggingFace] Loading tokenizer from: {local_model_path}")

        tokenizer = AutoTokenizer.from_pretrained(
            local_model_path,
            trust_remote_code=self.trust_remote_code,
            token=self.hf_token,
        )

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        torch_dtype = self._get_torch_dtype()

        model_kwargs = {
            "trust_remote_code": self.trust_remote_code,
            "device_map": "auto",
            "low_cpu_mem_usage": True,
            "token": self.hf_token,
        }

        if torch.cuda.is_available() and self.load_in_4bit:
            from transformers import BitsAndBytesConfig

            quantization_config = BitsAndBytesConfig(
                load_in_4bit=False,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch_dtype,
                bnb_4bit_use_double_quant=True,
            )

            model_kwargs["quantization_config"] = quantization_config
        else:
            model_kwargs["torch_dtype"] = torch_dtype

        print("[HuggingFace] Loading model. This can take a few minutes the first time.")

        model = AutoModelForCausalLM.from_pretrained(
            local_model_path,
            **model_kwargs
        )

        model.eval()

        HuggingFace._loaded_model_id = self.model_id
        HuggingFace._tokenizer = tokenizer
        HuggingFace._model = model
        HuggingFace._local_model_path = local_model_path

        print("[HuggingFace] Model loaded successfully.")

    def _format_prompt(self, conversation: Conversation):
        tokenizer = HuggingFace._tokenizer

        messages = []
        for msg in conversation.get_messages():
            role = msg.get("role", "user")
            content = msg.get("content", "")

            if role not in ["system", "user", "assistant"]:
                role = "user"

            messages.append({"role": role, "content": content})

        # Prefer the model's own chat template when available.
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            # Generic fallback for non-chat-template models.
            prompt_parts = []
            for msg in messages:
                prompt_parts.append(f"{msg['role'].upper()}:\n{msg['content']}\n")
            prompt_parts.append("ASSISTANT:\n")
            return "\n".join(prompt_parts)

    def _clean_output(self, text):
        text = text.strip()

        # Helpful for reasoning models that emit <think>...</think>.
        if self.strip_thinking:
            text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

        return text

    def generate(self, conversation: Conversation, num_choices=1):
        import torch

        tokenizer = HuggingFace._tokenizer
        model = HuggingFace._model

        prompt = self._format_prompt(conversation)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=False,
        )

        # With device_map="auto", use the first model parameter's device.
        first_device = next(model.parameters()).device
        inputs = {k: v.to(first_device) for k, v in inputs.items()}

        do_sample = self.temperature is not None and self.temperature > 0

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=do_sample,
                temperature=self.temperature if do_sample else None,
                top_p=self.top_p if do_sample else None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        generated_ids = outputs[0][prompt_len:]

        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return self._clean_output(text)


################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################

def find_verilog_modules(markdown_string, module_name='top_module'):
    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'
    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)
    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2

    if not module_matches:
        return []

    return module_matches


def write_code_blocks_to_file(markdown_string, module_name, filename):
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print("No Verilog modules found in response")
        exit(3)

    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')


def generate_verilog(conv, model_type=None, model_id=""):
    model_type = model_type or os.environ.get("MODEL", "ChatGPT")
    selected_model_id = model_id or model_choice

    if model_type == "ChatGPT":
        model = ChatGPT(model_id=selected_model_id)
    elif model_type == "Claude":
        model = Claude(model_id=selected_model_id)
    elif model_type == "Gemini":
        model = Gemini(model_id=selected_model_id)
    elif model_type == "HuggingFace":
        model = HuggingFace(model_id=selected_model_id)
    else:
        raise ValueError(
            f"Invalid model type: {model_type}. "
            "Expected one of: ChatGPT, Claude, Gemini, HuggingFace."
        )

    return model.generate(conv)

In [ ]:
#@title Feedback Loop

import subprocess
import sys
import os
import time
import numpy as np


def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []
    for p in file_list:
        if not p:
            continue

        try:
            with open(p, "r") as f:
                text = f.read()
                buf.append(text)
                if not text.endswith("\n"):
                    buf.append("\n")
        except FileNotFoundError:
            print(f"[Warning] Previous module file not found and will be skipped: {p}")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the end of long logs, where compiler errors usually appear."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    tail = s[-max_chars:]
    return "[...truncated...]\n" + tail


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a way that works for API models and local HF models.

    ChatGPT/Gemini/HuggingFace can use a system role.
    Claude path in the earlier utility cell maps unsupported roles to user,
    but we keep Claude as user here to avoid role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for your Verilog testbenches.

    Accepts:
      - "All tests passed"
      - "passed!"
      - "Test x passed"

    Rejects:
      - "failed"
      - "Some tests failed"
      - non-empty fatal/error-like stderr
    """
    out = (stdout_text or "")
    err = (stderr_text or "")
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "error running",
        "fatal",
        "syntax error",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


def verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    outdir="",
    log=None,
    prev_modules=None,
    model_id="",
):
    """
    Generate/fix a Verilog module with an LLM, compiling against previously-generated modules.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Key behaviors:
      - The LLM is asked to output ONLY the current module.
      - For compilation, this concatenates:
            [prev_modules...] + [current module]
        into a temporary combined file.
      - After an error, repair iterations only resend:
            (a) previous module version
            (b) most recent compiler/simulation feedback
        to keep prompt size controlled.
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]
    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    if outdir is None:
        outdir = ""

    outdir = outdir.strip()

    if outdir != "":
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir == "":
            return filename
        return os.path.join(outdir, filename)

    # Normalize prev_modules into a list of paths.
    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    # Read the testbench so the model can target the test expectations.
    tb_text = ""
    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "Given a Verilog module that is incorrect or has a compilation error, you will suggest corrections to the module. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete corrected Verilog module, from module declaration through endmodule. "
        "Use Verilog-2001 only unless explicitly told otherwise."
    )

    def _new_conv_full_context():
        """Initial attempt: include testbench and design prompt."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile/simulate your module. "
                "Do NOT generate a testbench or modify it. Only use it to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n",
            )

        conv.add_message("user", design_prompt)
        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        """Repair attempts: only previous module plus most recent error feedback."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog module so it compiles and passes the testbench. "
            "Output ONLY the full corrected module. "
            "Do not output explanations. "
            "Do not output a testbench. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)
        return conv

    conv = _new_conv_full_context()

    success = False
    timeout = False

    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    module_file = _outpath(module + ".v")
    combined_file = _outpath(module + "__combined.v")
    sim_output_file = _outpath(module)

    status = ""
    last_module_text = ""
    last_error_text = ""

    while not (success or timeout):
        print(f"\n========== Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        # Generate or repair only the current module.
        response = generate_verilog(conv, model_type, model_id=model_id)

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        conv.add_message("assistant", response)

        # Write raw model response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_" + str(iterations) + ".txt")
        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and write just the current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            last_module_text = response
            conv = _new_conv_repair_only(last_module_text, last_error_text)

            if iterations >= max_iterations:
                timeout = True

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)
            continue

        # Build concatenated source for compilation:
        # previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(module_file, "r") as f:
            curr_text = f.read()

        last_module_text = curr_text

        with open(combined_file, "w") as f:
            f.write(prev_text)
            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")
            f.write(curr_text)
            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # Compile with iVerilog.
        compile_cmd = [
            "iverilog",
            "-o",
            sim_output_file,
            combined_file,
            testbench,
        ]

        proc = subprocess.run(
            compile_cmd,
            capture_output=True,
            text=True,
        )

        success = False

        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)

            last_error_text = (
                "iverilog failed.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        elif proc.stderr.strip() != "":
            # Treat warnings as repair feedback.
            # This keeps generated modules cleaner for later synthesis/PPA.
            status = "Warnings compiling testbench"
            print(status)

            last_error_text = (
                "iverilog produced warnings.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        else:
            # Run simulation.
            run_cmd = ["vvp", sim_output_file]

            proc = subprocess.run(
                run_cmd,
                capture_output=True,
                text=True,
            )

            result_stdout = proc.stdout or ""
            result_stderr = proc.stderr or ""

            print(result_stdout)

            if proc.returncode != 0:
                status = "Error running testbench"
                print(status)

                last_error_text = (
                    "vvp simulation failed.\n\n"
                    "Command:\n"
                    + " ".join(run_cmd)
                    + "\n\nstdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            elif not _simulation_passed(result_stdout, result_stderr):
                status = "Testbench did not report success"
                print(status)

                last_error_text = (
                    "Simulation completed, but the output did not indicate success.\n\n"
                    "stdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            else:
                status = "Testbench ran successfully"
                print(status)

                last_error_text = ""
                success = True

        # Write bounded iteration log.
        with open(_outpath("log_iter_" + str(iterations) + ".txt"), "w") as file:
            file.write("\n".join(str(i) for i in conv.get_messages()))
            file.write("\n\nIteration status: " + status + "\n")

            if last_error_text:
                file.write("\nMost recent feedback:\n")
                file.write(_truncate_text(last_error_text))

        if not success:
            # Keep future repair prompts small.
            conv = _new_conv_repair_only(last_module_text, last_error_text)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1

        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print("\n========== Feedback Loop Complete ==========")
    print("Final status: ", status)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return (
        np.sum(timelist_total),
        np.sum(timelist_gen),
        np.sum(timelist_error),
    )

In [ ]:
#@title Hierarchical Loop

import os
import re
import numpy as np


def _normalize_to_list(x):
    """Normalize None/string/list/tuple into a list."""
    if x is None or x == "":
        return []
    if isinstance(x, str):
        return [x]
    return list(x)


def _resolve_prev_module_path(item):
    """
    Resolve previous module reference into a .v path.

    Supported forms:
      - "mux2_1"              -> "./mux2_1/mux2_1.v"
      - "./mux2_1/mux2_1.v"   -> "./mux2_1/mux2_1.v"
      - ("mux2_1", "...", ...) -> "./mux2_1/mux2_1.v"
    """
    if isinstance(item, (list, tuple)):
        item = item[0]

    item = str(item)

    if item.endswith(".v"):
        return item

    return os.path.join("./" + item, item + ".v")


def _extract_verilog_interfaces(verilog_text):
    """
    Extract module declarations/interfaces from Verilog text.

    This is useful for hierarchical prompting because the current module usually
    only needs previous module names and port lists, not full implementations.
    """
    if not verilog_text:
        return ""

    # Match:
    #   module name (...);
    #   module name #(parameters...) (...);
    pattern = r'\bmodule\s+[A-Za-z_][A-Za-z0-9_]*\s*(?:#\s*\(.*?\)\s*)?\(.*?\)\s*;'

    matches = re.findall(pattern, verilog_text, flags=re.DOTALL)

    if not matches:
        return ""

    cleaned = []
    for m in matches:
        cleaned.append(m.strip() + "\nendmodule")

    return "\n\n".join(cleaned) + "\n"


def _read_prev_modules_for_prompt(prev_verilog_files, context_mode="interfaces", max_chars=24000):
    """
    Read previous modules for LLM context.

    context_mode:
      - "interfaces": include only module declarations, best for HF/local models.
      - "full": include full previous module bodies.
      - "none": include no previous code in the prompt.

    Compilation still receives the full previous .v files through verilog_loop(prev_modules=...).
    """
    if context_mode == "none":
        return ""

    chunks = []

    for p in prev_verilog_files:
        if not p:
            continue

        if not os.path.exists(p):
            print(f"[Warning] Previous module file not found for prompt context: {p}")
            continue

        with open(p, "r") as f:
            txt = f.read()

        if context_mode == "interfaces":
            txt = _extract_verilog_interfaces(txt)

        if txt.strip():
            chunks.append("// File: " + p + "\n" + txt.strip() + "\n")

    prev_text = "\n".join(chunks)

    if len(prev_text) > max_chars:
        # Keep the most recent context, because later modules tend to depend on recent modules.
        prev_text = "[...previous module context truncated...]\n" + prev_text[-max_chars:]

    return prev_text


def _call_verilog_loop_compatible(
    prompt,
    module,
    testbench,
    max_iterations,
    model,
    outdir,
    log,
    prev_verilog_files,
    model_id="",
):
    """
    Calls the updated verilog_loop with model_id support.
    Falls back to the older signature if needed.
    """
    try:
        return verilog_loop(
            prompt,
            module,
            testbench,
            max_iterations,
            model,
            outdir,
            log,
            prev_modules=prev_verilog_files,
            model_id=model_id,
        )
    except TypeError as e:
        if "model_id" not in str(e):
            raise

        print("[Warning] verilog_loop does not accept model_id. Falling back to old signature.")

        return verilog_loop(
            prompt,
            module,
            testbench,
            max_iterations,
            model,
            outdir,
            log,
            prev_modules=prev_verilog_files,
        )


def hier_gen(
    submods,
    prevsubmods="",
    max_iterations=10,
    model_type=None,
    model_id="",
    testbench_template="./{module}tb.v",
    prompt_prev_context_chars=24000,
    prev_context_mode="interfaces",
):
    """
    Hierarchically generate a design by building submodules in order.

    submods:
      list of tuples/lists:
        [
          (file_basename, human_name, io_string),
          ...
        ]

      Example:
        [
          ("mux2_1", "2-to-1 mux", "input wire a, input wire b, input wire sel, output wire y"),
          ("mux4_1", "4-to-1 mux", "input wire [3:0] in, input wire [1:0] sel, output wire y"),
        ]

    prevsubmods:
      Optional previous module references already generated before this call.

      Supported:
        - ""
        - None
        - "mux2_1"
        - ["mux2_1", "mux4_1"]
        - ["./mux2_1/mux2_1.v"]
        - previous submod tuples/lists

    model_type:
      Optional. If None, uses os.environ["MODEL"].

      Supported by the earlier utility cell:
        - "ChatGPT"
        - "Claude"
        - "Gemini"
        - "HuggingFace"

    model_id:
      Optional model override passed through to verilog_loop/generate_verilog.

      Example:
        model_id="nyu-dice-lab/VeriThoughts-Reasoning-14B"

    testbench_template:
      Default assumes testbenches are named:
        ./module_nametb.v

      You can also use:
        "./testbenches/{module}_tb.v"

    prev_context_mode:
      - "interfaces": best default, includes previous module port declarations only.
      - "full": includes full previous module code in prompt.
      - "none": does not include previous modules in prompt.

      Regardless of this setting, previous modules are still passed to iVerilog
      through verilog_loop(prev_modules=...).
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_context_modes = ["interfaces", "full", "none"]
    if prev_context_mode not in valid_context_modes:
        raise ValueError(
            f"Invalid prev_context_mode: {prev_context_mode}. "
            f"Expected one of: {valid_context_modes}"
        )

    totaltime = []
    gentime = []
    errortime = []

    prev_verilog_files = []

    # Add previously generated external modules, if any.
    for s in _normalize_to_list(prevsubmods):
        p = _resolve_prev_module_path(s)
        if os.path.exists(p):
            prev_verilog_files.append(p)
        else:
            print(f"[Warning] prevsubmod file does not exist and will be skipped: {p}")

    if not submods:
        print("[Warning] No submodules provided.")
        return (0, 0, 0)

    overall = submods[-1][1]

    for i in range(len(submods)):
        fcurr = submods[i][0]   # file/module basename
        curr = submods[i][1]    # human-readable name
        iocurr = submods[i][2]  # port list string

        outdir = "./" + fcurr
        os.makedirs(outdir, exist_ok=True)

        prev_text = _read_prev_modules_for_prompt(
            prev_verilog_files,
            context_mode=prev_context_mode,
            max_chars=prompt_prev_context_chars,
        )

        if i == 0 and len(prev_verilog_files) == 0:
            prompt = (
                "// We will be generating a "
                + overall
                + " hierarchically in Verilog.\n"
                "// IMPORTANT: Output ONLY the module requested below.\n"
                "// Do NOT generate any previous modules, future modules, explanations, or testbenches.\n"
                "// Use Verilog-2001 only.\n"
                "// Begin by generating: "
                + curr
                + "\n\n"
                "module "
                + fcurr
                + "("
                + iocurr
                + ");\n"
                "// Insert code here\n"
                "endmodule\n"
            )
        else:
            prompt = (
                "// We are generating a "
                + overall
                + " hierarchically in Verilog.\n"
                "// Use Verilog-2001 only.\n\n"
                "// Previously generated modules are shown below for reference only.\n"
                "// You may instantiate them, but you must NOT repeat their definitions.\n\n"
                + prev_text
                + "\n"
                "// IMPORTANT: Output ONLY the new module requested below.\n"
                "// Do NOT repeat any previous modules.\n"
                "// Do NOT generate explanations, Markdown fences, or a testbench.\n"
                "// Now generate: "
                + curr
                + "\n\n"
                "module "
                + fcurr
                + "("
                + iocurr
                + ");\n"
                "// Insert code here\n"
                "endmodule\n"
            )

        module = fcurr
        testbench = testbench_template.format(module=fcurr, name=fcurr)
        model = model_type
        log = os.path.join(outdir, "log.txt")

        if not os.path.exists(testbench):
            print(f"[Warning] Testbench file does not exist yet: {testbench}")
            print("[Warning] verilog_loop will still run, but iVerilog will fail unless the file exists.")

        total, gen, error = _call_verilog_loop_compatible(
            prompt=prompt,
            module=module,
            testbench=testbench,
            max_iterations=max_iterations,
            model=model,
            outdir=outdir,
            log=log,
            prev_verilog_files=prev_verilog_files,
            model_id=model_id,
        )

        print("Module ", i + 1, " done: ", curr)

        totaltime.append(total)
        gentime.append(gen)
        errortime.append(error)

        # Add the current module file to the compilation context for subsequent modules.
        curr_module_file = os.path.join(outdir, module + ".v")

        if os.path.exists(curr_module_file):
            prev_verilog_files.append(curr_module_file)
        else:
            print(f"[Warning] Expected generated module file not found: {curr_module_file}")

    overall_total = np.sum(totaltime)
    overall_gen = np.sum(gentime)
    overall_error = np.sum(errortime)

    print("Overall Total time: ", overall_total)
    print("Overall Generation Time: ", overall_gen)
    print("Overall Error handling time: ", overall_error)

    return (overall_total, overall_gen, overall_error)

In [ ]:
#@title PPA Optimizer Loop

#@title PPA Optimization Loop (Yosys no-liberty + iVerilog correctness)

import subprocess
import sys
import os
import time
import math
import json
import re
import shutil
import numpy as np


################################################################################
### GENERAL HELPERS
################################################################################

def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []

    for p in file_list:
        if not p:
            continue

        if not os.path.exists(p):
            print(f"[Warning] Previous module file not found and will be skipped: {p}")
            continue

        with open(p, "r") as f:
            txt = f.read()

        buf.append(txt)

        if txt and not txt.endswith("\n"):
            buf.append("\n")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the tail of long logs."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    return "[...truncated...]\n" + s[-max_chars:]


def _shell_quote_for_yosys(path):
    """Quote filenames for use inside a .ys script."""
    return '"' + str(path).replace("\\", "\\\\").replace('"', '\\"') + '"'


def _safe_float(x, default=None):
    try:
        return float(x)
    except Exception:
        return default


def _safe_int(x, default=None):
    try:
        return int(x)
    except Exception:
        return default


def _first_regex(text, patterns, cast=float):
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE | re.MULTILINE | re.DOTALL)
        if m:
            try:
                return cast(m.group(1))
            except Exception:
                pass

    return None


def _normalize_clk_signal(clk_signal):
    """Treat None, '', 'None', 'none', etc. as no clock."""
    if clk_signal is None:
        return None

    s = str(clk_signal).strip()

    if s == "" or s.lower() == "none":
        return None

    return s


def _has_any_parsed_metric(metrics):
    if not metrics:
        return False

    keys = [
        "transistor_est",
        "area_cmos",
        "num_cells",
        "num_wire_bits",
        "timing_proxy",
    ]

    return any(metrics.get(k) is not None for k in keys)


def _is_verbose(verbose):
    return str(verbose).strip().lower() in ["1", "true", "yes", "y"]


def _write_text(path, text):
    with open(path, "w") as f:
        f.write(text)


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a provider-compatible way.

    ChatGPT, Gemini, and HuggingFace can use the system role through the
    Conversation abstraction from the previous utility cell.

    Claude is kept as user to avoid unsupported-role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _generate_verilog_compatible(conv, model_type, model_id=""):
    """
    Calls generate_verilog with model_id support when available.
    Falls back to the older two-argument signature if needed.
    """
    try:
        return generate_verilog(conv, model_type, model_id=model_id)
    except TypeError as e:
        if "model_id" not in str(e):
            raise

        print("[Warning] generate_verilog does not accept model_id. Falling back to old signature.")
        return generate_verilog(conv, model_type)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for Verilog testbenches.

    Accepts:
      - All tests passed
      - passed!
      - Test x passed
      - Test x passed!

    Rejects:
      - failed
      - Some tests failed
      - fatal/error markers
    """
    out = stdout_text or ""
    err = stderr_text or ""
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "fatal",
        "syntax error",
        "error:",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


################################################################################
### YOSYS PARSING
################################################################################

def _parse_yosys_stat_text(stat_text):
    """
    Parse Yosys `stat -tech cmos` output.

    Supports both styles:
      Older style:
        Number of wires: ...
        Number of cells: ...
        Chip area for module ...: ...

      Newer compact style:
        6 wires
        165 wire bits
        61 cells
        Estimated number of transistors: 722
    """
    metrics = {}

    metrics["num_wires"] = _first_regex(
        stat_text,
        [
            r"Number of wires:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+wires\s*$",
        ],
        int,
    )

    metrics["num_wire_bits"] = _first_regex(
        stat_text,
        [
            r"Number of wire bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+wire bits\s*$",
        ],
        int,
    )

    metrics["num_public_wires"] = _first_regex(
        stat_text,
        [
            r"Number of public wires:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+public wires\s*$",
        ],
        int,
    )

    metrics["num_public_wire_bits"] = _first_regex(
        stat_text,
        [
            r"Number of public wire bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+public wire bits\s*$",
        ],
        int,
    )

    metrics["num_memories"] = _first_regex(
        stat_text,
        [
            r"Number of memories:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+memories\s*$",
        ],
        int,
    )

    metrics["num_memory_bits"] = _first_regex(
        stat_text,
        [
            r"Number of memory bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+memory bits\s*$",
        ],
        int,
    )

    metrics["num_processes"] = _first_regex(
        stat_text,
        [
            r"Number of processes:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+processes\s*$",
        ],
        int,
    )

    metrics["num_cells"] = _first_regex(
        stat_text,
        [
            r"Number of cells:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+cells\s*$",
        ],
        int,
    )

    transistor_est = _first_regex(
        stat_text,
        [
            r"Estimated number of transistors:\s*([0-9]+(?:\.[0-9]+)?)",
            r"Chip area for module .*?:\s*([0-9]+(?:\.[0-9]+)?)",
        ],
        float,
    )

    metrics["transistor_est"] = transistor_est

    # Backward-compatible area-like proxy.
    metrics["area_cmos"] = transistor_est

    return metrics


def _parse_yosys_timeest_text(time_text):
    """
    Parse a timing proxy from Yosys timeest output.

    Returns None if no timing proxy can be found.
    """
    metrics = {}

    if re.search(r"No domain .* in module", time_text or "", flags=re.IGNORECASE):
        metrics["timing_proxy"] = None
        return metrics

    timing_proxy = _first_regex(
        time_text,
        [
            r"critical path.*?([0-9]+(?:\.[0-9]+)?)\s*AIG",
            r"([0-9]+(?:\.[0-9]+)?)\s*AIG nodes",
            r"([0-9]+(?:\.[0-9]+)?)\s*nodes.*critical path",
            r"Estimated.*?([0-9]+(?:\.[0-9]+)?)",
            r"Delay\s*[:=]\s*([0-9]+(?:\.[0-9]+)?)",
        ],
        float,
    )

    metrics["timing_proxy"] = timing_proxy

    return metrics


def _format_metrics(metrics):
    if metrics is None:
        return "None"

    parts = []

    if metrics.get("transistor_est") is not None:
        parts.append("transistor_est={:.0f}".format(metrics["transistor_est"]))
    elif metrics.get("area_cmos") is not None:
        parts.append("area_cmos={:.3f}".format(metrics["area_cmos"]))

    if metrics.get("timing_proxy") is not None:
        parts.append("timing_proxy={:.3f}".format(metrics["timing_proxy"]))

    if metrics.get("num_cells") is not None:
        parts.append("num_cells={}".format(metrics["num_cells"]))

    if metrics.get("num_wire_bits") is not None:
        parts.append("wire_bits={}".format(metrics["num_wire_bits"]))

    return ", ".join(parts) if parts else "No parsed metrics"


################################################################################
### YOSYS ESTIMATION
################################################################################

def yosys_estimate_no_liberty(verilog_file, top_module, clk_signal=None, outdir=""):
    """
    Run a coarse no-liberty Yosys estimate.

    Uses:
      - stat -top <top> -tech cmos
      - optional timeest -clk <clk>

    Returns:
      {
        "ok": bool,
        "metrics": dict,
        "stat_text": str,
        "time_text": str,
        "yosys_stdout": str,
        "yosys_stderr": str,
        "error": str
      }
    """
    if outdir:
        os.makedirs(outdir, exist_ok=True)

    clk_signal = _normalize_clk_signal(clk_signal)

    script_path = os.path.join(outdir, "ppa_estimate.ys") if outdir else "ppa_estimate.ys"
    stat_path = os.path.join(outdir, "ppa_stat.txt") if outdir else "ppa_stat.txt"
    time_path = os.path.join(outdir, "ppa_time.txt") if outdir else "ppa_time.txt"

    ys = []
    ys.append("read_verilog {}".format(_shell_quote_for_yosys(verilog_file)))
    ys.append("hierarchy -check -top {}".format(top_module))
    ys.append("proc")
    ys.append("opt")
    ys.append("memory")
    ys.append("opt")
    ys.append("fsm")
    ys.append("opt")
    ys.append("techmap")
    ys.append("opt")
    ys.append("clean")
    ys.append(
        "tee -o {} stat -top {} -tech cmos".format(
            _shell_quote_for_yosys(stat_path),
            top_module,
        )
    )

    if clk_signal is not None:
        ys.append("flatten")
        ys.append(
            "tee -o {} timeest -clk {}".format(
                _shell_quote_for_yosys(time_path),
                clk_signal,
            )
        )

    with open(script_path, "w") as f:
        f.write("\n".join(ys) + "\n")

    proc = subprocess.run(
        ["yosys", "-s", script_path],
        capture_output=True,
        text=True,
    )

    stdout = proc.stdout or ""
    stderr = proc.stderr or ""

    result = {
        "ok": False,
        "metrics": {},
        "stat_text": "",
        "time_text": "",
        "yosys_stdout": stdout,
        "yosys_stderr": stderr,
        "error": "",
    }

    if proc.returncode != 0:
        result["error"] = "Yosys failed.\nstdout:\n{}\nstderr:\n{}".format(stdout, stderr)
        return result

    stat_text = ""
    time_text = ""

    if os.path.exists(stat_path):
        with open(stat_path, "r") as f:
            stat_text = f.read()

    if os.path.exists(time_path):
        with open(time_path, "r") as f:
            time_text = f.read()

    metrics = _parse_yosys_stat_text(stat_text)
    metrics.update(_parse_yosys_timeest_text(time_text))

    if metrics.get("transistor_est") is not None:
        metrics["power_proxy"] = metrics["transistor_est"]
    else:
        metrics["power_proxy"] = metrics.get("num_cells")

    result["ok"] = True
    result["metrics"] = metrics
    result["stat_text"] = stat_text
    result["time_text"] = time_text

    if not _has_any_parsed_metric(metrics):
        result["error"] = "Yosys ran successfully, but no metrics were parsed."

    return result


################################################################################
### PPA SCORING
################################################################################

def _component_ratio(candidate_value, baseline_value):
    if candidate_value is None or baseline_value is None:
        return None

    if baseline_value == 0:
        return None

    return float(candidate_value) / float(baseline_value)


def _relative_ppa_score(
    candidate_metrics,
    baseline_metrics,
    area_weight=0.60,
    timing_weight=0.30,
    power_weight=0.10,
):
    """
    Lower is better.

    Uses relative ratios against the current best correct design.
    """
    used_weight = 0.0
    score = 0.0

    cand_area = candidate_metrics.get("transistor_est", candidate_metrics.get("area_cmos"))
    base_area = baseline_metrics.get("transistor_est", baseline_metrics.get("area_cmos"))

    area_ratio = _component_ratio(cand_area, base_area)

    if area_ratio is not None:
        score += area_weight * area_ratio
        used_weight += area_weight

    timing_ratio = _component_ratio(
        candidate_metrics.get("timing_proxy"),
        baseline_metrics.get("timing_proxy"),
    )

    if timing_ratio is not None:
        score += timing_weight * timing_ratio
        used_weight += timing_weight

    power_ratio = _component_ratio(
        candidate_metrics.get("power_proxy"),
        baseline_metrics.get("power_proxy"),
    )

    if power_ratio is not None:
        score += power_weight * power_ratio
        used_weight += power_weight

    if used_weight == 0.0:
        return math.inf

    return score / used_weight


def _is_better_ppa(candidate_metrics, best_metrics, min_improvement=0.0025):
    """
    Returns True only when candidate has parsed metrics and is better.
    """
    if not _has_any_parsed_metric(candidate_metrics):
        return False

    if best_metrics is None or not _has_any_parsed_metric(best_metrics):
        return True

    cand_score = _relative_ppa_score(candidate_metrics, best_metrics)

    # Best-vs-itself is effectively 1.0, so lower is better.
    return cand_score < (1.0 - min_improvement)


################################################################################
### IVERILOG CHECK
################################################################################

def _compile_and_run_iverilog(module_name, combined_file, testbench, outdir):
    """
    Return:
      {
        "ok": bool,
        "status": str,
        "error_text": str,
        "stdout": str,
        "stderr": str
      }
    """
    exe_path = os.path.join(outdir, module_name) if outdir else module_name

    proc = subprocess.run(
        ["iverilog", "-o", exe_path, combined_file, testbench],
        capture_output=True,
        text=True,
    )

    if proc.returncode != 0:
        return {
            "ok": False,
            "status": "Error compiling testbench",
            "error_text": (
                "iverilog failed.\n"
                "Command:\n"
                + " ".join(["iverilog", "-o", exe_path, combined_file, testbench])
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            ),
            "stdout": proc.stdout or "",
            "stderr": proc.stderr or "",
        }

    warn_text = ""

    if proc.stderr and proc.stderr.strip():
        warn_text = (
            "iverilog produced warnings.\n"
            "stderr:\n"
            + (proc.stderr or "")
            + "\nstdout:\n"
            + (proc.stdout or "")
        )

    sim = subprocess.run(
        ["vvp", exe_path],
        capture_output=True,
        text=True,
    )

    result_stdout = sim.stdout or ""
    result_stderr = sim.stderr or ""

    if sim.returncode != 0:
        return {
            "ok": False,
            "status": "Error running testbench",
            "error_text": (
                "vvp simulation failed.\n"
                "stdout:\n"
                + result_stdout
                + "\nstderr:\n"
                + result_stderr
            ),
            "stdout": result_stdout,
            "stderr": result_stderr,
        }

    if not _simulation_passed(result_stdout, result_stderr):
        return {
            "ok": False,
            "status": "Testbench did not report success",
            "error_text": (
                "Simulation completed, but the output did not indicate success.\n"
                "stdout:\n"
                + result_stdout
                + "\nstderr:\n"
                + result_stderr
            ),
            "stdout": result_stdout,
            "stderr": result_stderr,
        }

    return {
        "ok": True,
        "status": "Testbench ran successfully",
        "error_text": warn_text,
        "stdout": result_stdout,
        "stderr": result_stderr,
    }


################################################################################
### MAIN PPA LOOP
################################################################################

def ppa_verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    clk_signal=None,
    outdir="",
    log=None,
    prev_modules=None,
    verbose="0",
    model_id="",
    keep_best_as_module=True,
):
    """
    Generate functionally correct Verilog, then iteratively improve coarse no-liberty
    PPA using Yosys estimates while preserving correctness.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Assumptions:
      - generate_verilog(conv, model_type, model_id=...) exists from the utility cell
      - Conversation(log_file=...) exists
      - write_code_blocks_to_file(response, module, path) exists
      - yosys, iverilog, and vvp are in PATH

    Strategy:
      1. Generate a candidate.
      2. Check correctness with iVerilog/vvp.
      3. If correct, estimate PPA with Yosys.
      4. Keep the best correct candidate seen so far.
      5. Ask the LLM to improve the best correct candidate further.
      6. If a candidate fails, repair using only the failed module plus latest error.
    """
    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]

    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    outdir = "" if outdir is None else str(outdir).strip()

    if outdir:
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir:
            return os.path.join(outdir, filename)
        return filename

    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    tb_text = ""

    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog-2001 code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete Verilog module from module declaration through endmodule. "
        "Preserve the exact module name and port interface. "
        "Preserve functionality required by the testbench. "
        "Optimize for lower coarse PPA under Yosys without a liberty file: "
        "reduce logic depth, reduce unnecessary cells, share repeated logic, simplify control, "
        "avoid redundant muxing, avoid unnecessary intermediate signals, and keep widths disciplined."
    )

    def _new_conv_initial():
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile and simulate your module. "
                "Do NOT generate a testbench or modify it. Use it only to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n"
            )

        conv.add_message(
            "user",
            design_prompt
            + "\n\n"
            "Also optimize the module for better coarse Yosys-estimated PPA without a liberty file. "
            "Output ONLY the full Verilog-2001 module. "
            "Do not use Markdown fences."
        )

        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog-2001 module so it compiles and passes the testbench. "
            "Preserve the exact interface and intended behavior. "
            "Keep PPA reasonably optimized, but correctness is the priority for this step. "
            "Output ONLY the full corrected module. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation/Yosys output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)

        return conv

    def _new_conv_optimize_from_best(best_module_text, best_metrics, last_candidate_metrics=None):
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Starting from the following functionally correct Verilog-2001 module, "
            "produce a new candidate that preserves the exact same behavior and interface "
            "but improves coarse Yosys-estimated PPA without a liberty file.\n\n"
            "Current best metrics:\n"
            + _format_metrics(best_metrics)
            + "\n\n"
        )

        if last_candidate_metrics is not None:
            prompt += (
                "Most recent candidate metrics:\n"
                + _format_metrics(last_candidate_metrics)
                + "\n\n"
            )

        prompt += (
            "Optimization priorities:\n"
            "1. Lower stat -tech cmos transistor_est / area proxy\n"
            "2. Lower timing proxy, if available\n"
            "3. Fewer cells and wire bits\n\n"
            "Rules:\n"
            "- Do not add helper modules.\n"
            "- Do not change functionality.\n"
            "- Do not change the module name or port interface.\n"
            "- Use Verilog-2001 only.\n"
            "- Output ONLY the full module.\n"
            "- Do not use Markdown fences.\n\n"
            "Current best module:\n"
            "```verilog\n"
            + best_module_text
            + "\n```\n"
        )

        conv.add_message("user", prompt)

        return conv

    # Timing
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    # Files
    module_file = _outpath(module + ".v")
    best_module_file = _outpath(module + "_best.v")
    combined_file = _outpath(module + "__combined.v")

    # Best-so-far state
    best_module_text = None
    best_metrics = None
    best_iter = None

    # Latest state
    conv = _new_conv_initial()
    last_error_text = ""
    last_candidate_metrics = None

    iterations = 0
    timeout = False

    while not timeout:
        print(f"\n========== PPA Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        response = _generate_verilog_compatible(
            conv,
            model_type,
            model_id=model_id,
        )

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        try:
            conv.add_message("assistant", response)
        except Exception:
            pass

        # Save raw response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_{:03d}.txt".format(iterations))

        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and persist current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            conv = _new_conv_repair_only(response, last_error_text)

            candidate_metrics = None

            log_path = _outpath("log_iter_{:03d}.txt".format(iterations))
            with open(log_path, "w") as f:
                f.write("Iteration status: {}\n".format(status))
                f.write("\nMost recent feedback:\n")
                f.write(_truncate_text(last_error_text))

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)

            if iterations >= max_iterations:
                timeout = True

            continue

        with open(module_file, "r") as f:
            curr_text = f.read()

        # Concatenate previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(combined_file, "w") as f:
            f.write(prev_text)

            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")

            f.write(curr_text)

            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # 1. Functional check with iVerilog/vvp.
        iverilog_result = _compile_and_run_iverilog(
            module_name=module,
            combined_file=combined_file,
            testbench=testbench,
            outdir=outdir,
        )

        status = iverilog_result["status"]
        accepted = False
        candidate_metrics = None

        if not iverilog_result["ok"]:
            last_error_text = iverilog_result["error_text"]
            conv = _new_conv_repair_only(curr_text, last_error_text)

        else:
            # 2. Coarse PPA estimate with Yosys.
            ppa_dir = _outpath("iter_{:03d}_ppa".format(iterations))
            os.makedirs(ppa_dir, exist_ok=True)

            ppa_result = yosys_estimate_no_liberty(
                verilog_file=combined_file,
                top_module=module,
                clk_signal=clk_signal,
                outdir=ppa_dir,
            )

            if _is_verbose(verbose):
                print("YOSYS OK:", ppa_result["ok"])
                print("YOSYS STDOUT HEAD:\n", (ppa_result["yosys_stdout"] or "")[:3000])
                print("YOSYS STDERR HEAD:\n", (ppa_result["yosys_stderr"] or "")[:3000])
                print("STAT TEXT HEAD:\n", repr((ppa_result["stat_text"] or "")[:1000]))
                print("TIME TEXT HEAD:\n", repr((ppa_result["time_text"] or "")[:1000]))
                print("PARSED METRICS:", ppa_result["metrics"])

            if not ppa_result["ok"]:
                status = "Yosys estimation failed"
                last_error_text = ppa_result["error"]
                conv = _new_conv_repair_only(curr_text, last_error_text)

            else:
                candidate_metrics = ppa_result["metrics"]
                last_candidate_metrics = candidate_metrics

                if not _has_any_parsed_metric(candidate_metrics):
                    status = "Yosys ran but returned no parsable metrics"

                    if best_module_text is not None and best_metrics is not None:
                        conv = _new_conv_optimize_from_best(
                            best_module_text=best_module_text,
                            best_metrics=best_metrics,
                            last_candidate_metrics=None,
                        )
                    else:
                        conv = _new_conv_initial()

                elif _is_better_ppa(candidate_metrics, best_metrics):
                    best_module_text = curr_text
                    best_metrics = candidate_metrics
                    best_iter = iterations
                    accepted = True

                    _write_text(best_module_file, best_module_text)

                    status = "Accepted new best candidate"

                    conv = _new_conv_optimize_from_best(
                        best_module_text=best_module_text,
                        best_metrics=best_metrics,
                        last_candidate_metrics=candidate_metrics,
                    )

                else:
                    status = "Candidate correct but did not improve best PPA"

                    conv = _new_conv_optimize_from_best(
                        best_module_text=best_module_text if best_module_text is not None else curr_text,
                        best_metrics=best_metrics if best_metrics is not None else candidate_metrics,
                        last_candidate_metrics=candidate_metrics,
                    )

        # Write iteration log.
        log_path = _outpath("log_iter_{:03d}.txt".format(iterations))

        with open(log_path, "w") as f:
            try:
                f.write("\n".join(str(i) for i in conv.get_messages()))
            except Exception:
                f.write("(Conversation logging unavailable)\n")

            f.write("\n\nIteration status: {}\n".format(status))

            if candidate_metrics is not None:
                f.write("Candidate metrics: {}\n".format(_format_metrics(candidate_metrics)))

            if best_metrics is not None:
                f.write("Best metrics so far: {}\n".format(_format_metrics(best_metrics)))
                f.write("Best iteration so far: {}\n".format(best_iter))

            if last_error_text:
                f.write("\nMost recent feedback:\n")
                f.write(_truncate_text(last_error_text))

        print("Iteration {}: {}".format(iterations, status))

        if candidate_metrics is not None:
            print("  Candidate metrics:", _format_metrics(candidate_metrics))

        if best_metrics is not None:
            print("  Best metrics so far:", _format_metrics(best_metrics))
            print("  Best iteration so far:", best_iter)

        iterations += 1

        end_time = time.time()

        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

        if iterations >= max_iterations:
            timeout = True

    # Optional: leave module.v as the best candidate, not the last failed/non-improved candidate.
    if keep_best_as_module and best_module_text is not None:
        _write_text(module_file, best_module_text)
        print(f"Best candidate copied to: {module_file}")

    print("\n========== PPA Optimization Complete ==========")
    print("Best file: ", best_module_file if best_module_text is not None else None)
    print("Best metrics: ", _format_metrics(best_metrics) if best_metrics is not None else None)
    print("Best iteration: ", best_iter)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return {
        "best_file": best_module_file if best_module_text is not None else None,
        "best_metrics": best_metrics,
        "best_iteration": best_iter,
        "iterations": iterations,
        "total_time": float(np.sum(timelist_total)),
        "generation_time": float(np.sum(timelist_gen)),
        "error_handling_time": float(np.sum(timelist_error)),
    }

# Setting the API Key

In [ ]:
### Defaults to Claude, just change the model in the Select Model cell in Initial Setup & choose your provider here
### Or, choose a Hugging Face model and set "MODEL" to "HuggingFace"

# [HW PATCH] The key is read from a Colab secret named OPENAI_API_KEY (key icon in the left sidebar),
# falling back to a hidden prompt. Nothing is stored in the notebook.
from getpass import getpass
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
assert os.environ["OPENAI_API_KEY"].startswith("sk-"), "OPENAI_API_KEY not set"
print("OpenAI key loaded (", len(os.environ["OPENAI_API_KEY"]), "chars )")

#os.environ['CLAUDE_API_KEY'] = "API key here"
#os.environ['GEMINI_API_KEY'] ="API key here"
os.environ["MODEL"] = "ChatGPT"   # [HW PATCH]
#os.environ["MODEL"] = "Claude"
#os.environ["MODEL"] = "Gemini"
#os.environ["MODEL"] = "HuggingFace"


#Mux Hierarchy Example

In [ ]:
#@title Pull Testbenches

#From the ROME GitHub Repo

!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/mux/mux2_1tb.v ./mux2_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux4_1tb.v ./mux4_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux8_1tb.v ./mux8_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux16_1tb.v ./mux16_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux32_1tb.v ./mux32_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux64_1tb.v ./mux64_1tb.v

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

In [ ]:
hier_gen(submodules)

In [ ]:
#@title Run PPA optimizer on hierarchical mux design
import os
import json

submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

# Adjust these if your testbench filenames differ.
tb_paths = {
    "mux2_1":  "./mux2_1tb.v",
    "mux4_1":  "./mux4_1tb.v",
    "mux8_1":  "./mux8_1tb.v",
    "mux16_1": "./mux16_1tb.v",
    "mux32_1": "./mux32_1tb.v",
    "mux64_1": "./mux64_1tb.v",
}

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./mux_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type=os.environ["MODEL"],   # [HW PATCH] use the selected backend
        clk_signal=None,   # all of these mux modules are combinational
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,         # set to 0 if you want less diagnostic output
    )

    results[filename] = result

    # Prefer the best optimized file if one was produced.
    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        # Fallback to the last generated module file in that run directory.
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path

In [ ]:
#@title Helpers to show ROME results (final simulation, iteration history, prompts)
import glob, subprocess, os, re

def rome_iterations(module):
    """Status of every ROME iteration for a module, from the log_iter_N.txt files."""
    rows = []
    for p in sorted(glob.glob(f"./{module}/log_iter_*.txt"), key=lambda x: int(re.findall(r"(\d+)", x)[-1])):
        txt = open(p).read()
        m = re.search(r"Iteration status: (.*)", txt)
        rows.append((os.path.basename(p), m.group(1).strip() if m else "?"))
    return rows

def final_sim(module, testbench, rtl_files=None):
    """Recompile the final RTL of a module with its FIXED testbench and show the simulation output."""
    rtl_files = rtl_files or [f"./{module}/{module}__combined.v"]
    exe = f"./{module}/{module}_final"
    cmd = ["iverilog", "-o", exe] + rtl_files + [testbench]
    print("$ " + " ".join(cmd))
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stderr); return False
    print(f"$ vvp {exe}")
    p = subprocess.run(["vvp", exe], capture_output=True, text=True)
    out = p.stdout + p.stderr
    print(out)
    with open(f"./{module}/final_sim.txt", "w") as f:
        f.write(" ".join(cmd) + "\n" + "vvp " + exe + "\n" + out)
    ok = _simulation_passed(p.stdout, p.stderr)
    print("=> PASS" if ok else "=> FAIL", "\n")
    return ok

def show_prompt(module):
    """The design prompt ROME sent for this module (from the conversation log)."""
    txt = open(f"./{module}/log.txt").read()
    m = re.search(r"user: (// We (?:will be|are) generating.*?)(?=\n(?:assistant|user|system): |\Z)", txt, re.DOTALL)
    print(m.group(1) if m else "(prompt not found in log)")

print("helpers ready")

# Part I: original ROME mux hierarchy, final results
The cells above ran the unmodified ROME loop (`hier_gen`) and the PPA optimizer. The next cells recompile each
module's final RTL with the supplied fixed testbench and show the simulation output ending in `passed!`,
list how many ROME iterations each module needed, and summarize the PPA proxy results.

In [ ]:
#@title Part I: final simulation output for every mux module + ROME iteration history
mux_modules = ["mux2_1", "mux4_1", "mux8_1", "mux16_1", "mux32_1", "mux64_1"]
part1_status = {}
for m in mux_modules:
    print("=" * 80 + f"\n{m}\n" + "=" * 80)
    for fname, status in rome_iterations(m):
        print(f"  {fname}: {status}")
    part1_status[m] = final_sim(m, f"./{m}tb.v")
print("Part I summary:", part1_status)

In [ ]:
#@title Part I: PPA proxy results (Yosys stat -tech cmos, no liberty) from the PPA optimizer run
import json
ppa = json.load(open("./mux_ppa_runs/summary.json"))
print(f"{'module':<10}{'best iter':>10}{'transistor_est':>16}{'num_cells':>11}{'wire_bits':>11}{'timing_proxy':>14}")
for m in mux_modules:
    r = ppa[m]; bm = r.get("best_metrics") or {}
    print(f"{m:<10}{str(r.get('best_iteration')):>10}{str(bm.get('transistor_est')):>16}{str(bm.get('num_cells')):>11}"
          f"{str(bm.get('num_wire_bits')):>11}{str(bm.get('timing_proxy')):>14}")

# Part II: new hierarchical design `half_adder -> full_adder -> adder4 -> adder8`
Four modules, generated bottom-up with the same `hier_gen` loop. Each has its own self-checking baseline testbench
(written by hand, below). Every testbench computes the expected value from a reference expression
(`exp = a + b + cin`) inside a `check` task, prints `Test N passed` / `Test N FAILED` per check and `All tests passed!`
only when nothing failed. The `// === COVERAGE TESTS ===` marker is where Part III inserts coverage-guided tests;
the RTL loop never touches the testbench.

Interfaces:
- `half_adder(a, b -> sum, cout)`
- `full_adder(a, b, cin -> sum, cout)`: two `half_adder` instances
- `adder4([3:0] a, [3:0] b, cin -> [3:0] sum, cout)`: four `full_adder` instances, ripple carry
- `adder8([7:0] a, [7:0] b, cin -> [7:0] sum, cout)`: two `adder4` instances (top module)

In [ ]:
!mkdir -p adder_tb

In [ ]:
%%writefile adder_tb/half_adder_tb.v
`timescale 1ns/1ps
// Baseline self-checking testbench for half_adder.
// Expected values come from a reference expression, not hand-typed constants.
module tb_half_adder;
    reg  a, b;
    wire sum, cout;

    half_adder dut (.a(a), .b(b), .sum(sum), .cout(cout));

    integer testnum;
    integer fail_count;

    task check;
        input in_a, in_b;
        reg [1:0] exp;
        begin
            a = in_a; b = in_b;
            #10;
            exp = in_a + in_b;          // reference: {cout, sum}
            testnum = testnum + 1;
            if ({cout, sum} === exp)
                $display("Test %0d passed (a=%b b=%b sum=%b cout=%b)", testnum, a, b, sum, cout);
            else begin
                $display("Test %0d FAILED (a=%b b=%b expected {cout,sum}=%b got %b%b)",
                         testnum, a, b, exp, cout, sum);
                fail_count = fail_count + 1;
            end
        end
    endtask

    initial begin
        testnum = 0; fail_count = 0;
        a = 0; b = 0;

        // === BASELINE TESTS ===
        check(0, 0);
        check(0, 1);
        check(1, 0);
        check(1, 1);
        // === COVERAGE TESTS === (coverage-guided tests are inserted below this line)

        if (fail_count == 0)
            $display("All tests passed!");
        else
            $display("SUMMARY: %0d test(s) FAILED", fail_count);
        $finish;
    end
endmodule


In [ ]:
%%writefile adder_tb/full_adder_tb.v
`timescale 1ns/1ps
// Baseline self-checking testbench for full_adder.
module tb_full_adder;
    reg  a, b, cin;
    wire sum, cout;

    full_adder dut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    integer testnum;
    integer fail_count;

    task check;
        input in_a, in_b, in_cin;
        reg [1:0] exp;
        begin
            a = in_a; b = in_b; cin = in_cin;
            #10;
            exp = in_a + in_b + in_cin;   // reference: {cout, sum}
            testnum = testnum + 1;
            if ({cout, sum} === exp)
                $display("Test %0d passed (a=%b b=%b cin=%b sum=%b cout=%b)", testnum, a, b, cin, sum, cout);
            else begin
                $display("Test %0d FAILED (a=%b b=%b cin=%b expected {cout,sum}=%b got %b%b)",
                         testnum, a, b, cin, exp, cout, sum);
                fail_count = fail_count + 1;
            end
        end
    endtask

    initial begin
        testnum = 0; fail_count = 0;
        a = 0; b = 0; cin = 0;

        // === BASELINE TESTS ===
        check(0, 0, 0);
        check(0, 1, 0);
        check(1, 0, 0);
        check(1, 1, 0);
        check(1, 1, 1);
        // === COVERAGE TESTS === (coverage-guided tests are inserted below this line)

        if (fail_count == 0)
            $display("All tests passed!");
        else
            $display("SUMMARY: %0d test(s) FAILED", fail_count);
        $finish;
    end
endmodule


In [ ]:
%%writefile adder_tb/adder4_tb.v
`timescale 1ns/1ps
// Baseline self-checking testbench for adder4 (4-bit ripple-carry adder).
module tb_adder4;
    reg  [3:0] a, b;
    reg        cin;
    wire [3:0] sum;
    wire       cout;

    adder4 dut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    integer testnum;
    integer fail_count;

    task check;
        input [3:0] in_a, in_b;
        input       in_cin;
        reg   [4:0] exp;
        begin
            a = in_a; b = in_b; cin = in_cin;
            #10;
            exp = in_a + in_b + in_cin;   // reference: {cout, sum}
            testnum = testnum + 1;
            if ({cout, sum} === exp)
                $display("Test %0d passed (a=%h b=%h cin=%b sum=%h cout=%b)", testnum, a, b, cin, sum, cout);
            else begin
                $display("Test %0d FAILED (a=%h b=%h cin=%b expected {cout,sum}=%b got %b%b)",
                         testnum, a, b, cin, exp, cout, sum);
                fail_count = fail_count + 1;
            end
        end
    endtask

    initial begin
        testnum = 0; fail_count = 0;
        a = 0; b = 0; cin = 0;

        // === BASELINE TESTS ===
        check(4'h0, 4'h0, 0);
        check(4'h1, 4'h1, 0);
        check(4'h5, 4'hA, 0);
        check(4'h7, 4'h1, 0);
        check(4'h3, 4'h4, 0);
        check(4'h9, 4'h6, 0);
        // === COVERAGE TESTS === (coverage-guided tests are inserted below this line)

        if (fail_count == 0)
            $display("All tests passed!");
        else
            $display("SUMMARY: %0d test(s) FAILED", fail_count);
        $finish;
    end
endmodule


In [ ]:
%%writefile adder_tb/adder8_tb.v
`timescale 1ns/1ps
// Baseline self-checking testbench for adder8 (8-bit ripple-carry adder, top module).
module tb_adder8;
    reg  [7:0] a, b;
    reg        cin;
    wire [7:0] sum;
    wire       cout;

    adder8 dut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    integer testnum;
    integer fail_count;

    task check;
        input [7:0] in_a, in_b;
        input       in_cin;
        reg   [8:0] exp;
        begin
            a = in_a; b = in_b; cin = in_cin;
            #10;
            exp = in_a + in_b + in_cin;   // reference: {cout, sum}
            testnum = testnum + 1;
            if ({cout, sum} === exp)
                $display("Test %0d passed (a=%h b=%h cin=%b sum=%h cout=%b)", testnum, a, b, cin, sum, cout);
            else begin
                $display("Test %0d FAILED (a=%h b=%h cin=%b expected {cout,sum}=%b got %b%b)",
                         testnum, a, b, cin, exp, cout, sum);
                fail_count = fail_count + 1;
            end
        end
    endtask

    initial begin
        testnum = 0; fail_count = 0;
        a = 0; b = 0; cin = 0;

        // === BASELINE TESTS ===
        check(8'h00, 8'h00, 0);
        check(8'h01, 8'h01, 0);
        check(8'h0F, 8'h01, 0);
        check(8'h55, 8'h2A, 0);
        check(8'h12, 8'h34, 0);
        check(8'h80, 8'h7F, 0);
        check(8'h0F, 8'h0F, 0);
        check(8'h33, 8'h44, 0);
        // === COVERAGE TESTS === (coverage-guided tests are inserted below this line)

        if (fail_count == 0)
            $display("All tests passed!");
        else
            $display("SUMMARY: %0d test(s) FAILED", fail_count);
        $finish;
    end
endmodule


In [ ]:
#@title Part II: generate the adder hierarchy with the ROME loop (testbenches fixed)
adder_submodules = [
    ["half_adder", "half adder (sum = a XOR b, cout = a AND b)",
     "input wire a, input wire b, output wire sum, output wire cout"],
    ["full_adder", "full adder built from two half_adder instances plus an OR gate for the carry (instantiate half_adder; do not use the + operator)",
     "input wire a, input wire b, input wire cin, output wire sum, output wire cout"],
    ["adder4", "4-bit ripple-carry adder built from four full_adder instances (instantiate full_adder; do not use the + operator)",
     "input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout"],
    ["adder8", "8-bit ripple-carry adder built from two adder4 instances, low nibble then high nibble with the carry chained (instantiate adder4; do not use the + operator)",
     "input wire [7:0] a, input wire [7:0] b, input wire cin, output wire [7:0] sum, output wire cout"],
]
hier_gen(adder_submodules, testbench_template="./adder_tb/{module}_tb.v")

In [ ]:
#@title Part II: final simulation per adder module, iteration history, and the prompts used
adder_modules = ["half_adder", "full_adder", "adder4", "adder8"]
adder_rtl = [f"./{m}/{m}.v" for m in adder_modules]
part2_status = {}
for i, m in enumerate(adder_modules):
    print("=" * 80 + f"\n{m}\n" + "=" * 80)
    for fname, status in rome_iterations(m):
        print(f"  {fname}: {status}")
    part2_status[m] = final_sim(m, f"./adder_tb/{m}_tb.v", adder_rtl[: i + 1])
print("Part II summary:", part2_status)

print("\n" + "=" * 80 + "\nFinal prompt for the top module (adder8)\n" + "=" * 80)
show_prompt("adder8")
print("\n" + "=" * 80 + "\nFinal prompt for a representative submodule (full_adder)\n" + "=" * 80)
show_prompt("full_adder")
print("\n" + "=" * 80 + "\nGenerated RTL\n" + "=" * 80)
for p in adder_rtl:
    print(f"// ---- {p}\n" + open(p).read())

# Part III: integrate the iVerilog + Covered flow (`coverage_exp`) into ROME
The two cells below are my integration of the course `coverage_exp` flow. `run_coverage` reproduces the Makefile
steps (`iverilog -s <tb>` -> `vvp` -> strip the VCD `$comment` that Covered 0.7.10 rejects -> `covered score -t <top> -i <tb>.<dut>`
-> `covered report -m ltc -d s` / `-d v -i`), parses the reports and turns the misses into feedback text.
`coverage_loop` is the second feedback loop: with the RTL frozen, the LLM is given the current testbench plus the
uncovered targets and asked only for new `check(...)` calls, which are inserted at the marker. If a stronger testbench
fails, the testbench is frozen and `verilog_loop` (ROME) repairs the top module. Everything is saved under `coverage/`
plus `coverage_history.json`, and every RTL or TB change is appended to `iteration_log.json`.

Coverage is reported per **instance** (`covered report -i`), because in a purely structural hierarchy the
per-module view merges all instances of `half_adder`/`full_adder` and saturates immediately; the per-instance view
shows which carry paths and nibbles the testbench actually exercised.

In [ ]:
#@title Coverage integration: iVerilog + vvp + Covered (from coverage_exp)
# Student integration of the course coverage_exp flow into ROME.
# The flow follows coverage_exp/Makefile step by step:
#   iverilog -> vvp (VCD) -> strip the VCD comment Covered 0.7.10 cannot parse
#   -> covered score -> covered report (summary + detail) -> parse -> feedback for the LLM.
#
# The self-checking testbenches are NOT modified to add $dumpfile/$dumpvars. Instead the
# simulation is *wrapped*: a tiny extra module (cov_dump) is compiled next to the testbench
# and dumps the whole testbench hierarchy. This keeps "TB fixed during RTL repair" literally
# true: the same testbench file is used by ROME and by the coverage run.

import os
import re
import json
import shutil
import subprocess

COV_ROOT = "./coverage"
os.makedirs(COV_ROOT, exist_ok=True)

IVERILOG = shutil.which("iverilog")
VVP = shutil.which("vvp")
COVERED = os.environ.get("COVERED") or shutil.which("covered")
assert IVERILOG and VVP, "iverilog/vvp not found"
assert COVERED, "covered not found: run the install cell (apt-get install covered)"


def _run(cmd, cwd=None):
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    return proc.returncode, (proc.stdout or ""), (proc.stderr or "")


def _parse_tb(tb_text, dut_module):
    """Return (testbench module name, DUT instance name) from a testbench file."""
    m = re.search(r"\bmodule\s+([A-Za-z_][A-Za-z0-9_]*)\s*(?:\([^)]*\))?\s*;", tb_text)
    tb_top = m.group(1) if m else None
    m = re.search(r"\b" + re.escape(dut_module) + r"\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(", tb_text)
    inst = m.group(1) if m else None
    if not tb_top or not inst:
        raise ValueError("Could not find testbench module name or DUT instance name for " + dut_module)
    return tb_top, inst


def _clean_vcd(raw_path, clean_path):
    """Same transformation as the coverage_exp Makefile / demo notebook."""
    with open(raw_path, "r") as f:
        lines = f.read().splitlines(keepends=True)
    text = "".join(l for l in lines if not l.startswith("$comment "))
    text = text.replace("$timescale\n\t1ps\n$end\n", "$timescale 1ps $end\n")
    with open(clean_path, "w") as f:
        f.write(text)


def _count_tests(sim_stdout):
    """Number of individual checks reported by the testbench."""
    n = 0
    for line in sim_stdout.splitlines():
        if re.search(r"\btest\s*\d+\b.*\b(passed|failed)\b", line, re.IGNORECASE):
            n += 1
    return n


def _parse_summary(summary_text):
    """Parse the 'Accumulated' rows of a Covered summary report (line, toggle, logic)."""
    def section(name_regex):
        m = re.search(name_regex, summary_text)
        if not m:
            return ""
        rest = summary_text[m.end():]
        rest = re.sub(r"^[^\n]*\n~+\n", "", rest)          # skip the rest of the banner
        nxt = re.search(r"\n~{20,}", rest)
        return rest[:nxt.start()] if nxt else rest

    res = {}
    sec = section(r"LINE COVERAGE RESULTS")
    m = re.search(r"Accumulated\s+(\d+)/\s*(\d+)/\s*(\d+)\s+(\d+)%", sec)
    if m:
        res["line"] = {"hit": int(m.group(1)), "miss": int(m.group(2)), "total": int(m.group(3)), "pct": int(m.group(4))}

    sec = section(r"TOGGLE COVERAGE RESULTS")
    m = re.search(r"Accumulated\s+(\d+)/\s*(\d+)/\s*(\d+)\s+(\d+)%\s+(\d+)/\s*(\d+)/\s*(\d+)\s+(\d+)%", sec)
    if m:
        res["toggle01"] = {"hit": int(m.group(1)), "miss": int(m.group(2)), "total": int(m.group(3)), "pct": int(m.group(4))}
        res["toggle10"] = {"hit": int(m.group(5)), "miss": int(m.group(6)), "total": int(m.group(7)), "pct": int(m.group(8))}

    sec = section(r"COMBINATIONAL LOGIC COVERAGE RESULTS")
    m = re.search(r"Accumulated\s+(\d+)/\s*(\d+)/\s*(\d+)\s+(\d+)%", sec)
    if m:
        res["logic"] = {"hit": int(m.group(1)), "miss": int(m.group(2)), "total": int(m.group(3)), "pct": int(m.group(4))}
    return res


def _hex_to_int(s):
    m = re.match(r"(\d+)'h([0-9a-fA-F]+)", s)
    return (int(m.group(1)), int(m.group(2), 16)) if m else (None, None)


def _missing_bits(width, mask):
    return [i for i in range(width) if not (mask >> i) & 1]


def _parse_detail(detail_text):
    """
    Extract uncovered targets from a Covered verbose instance report:
      missed_lines : [(instance, line_no, text)]
      toggles      : [(instance, signal, missing_0to1_bits, missing_1to0_bits)]
      logic        : [(instance, line_no, expr_text, hit, total, missed_values)]
    """
    out = {"missed_lines": [], "toggles": [], "logic": []}

    def blocks(section_regex):
        m = re.search(section_regex, detail_text)
        if not m:
            return []
        rest = detail_text[m.end():]
        rest = re.sub(r"^[^\n]*\n~+\n", "", rest)          # skip the rest of the banner
        nxt = re.search(r"\n~{20,}", rest)
        rest = rest[:nxt.start()] if nxt else rest
        parts = re.split(r"\n\s*Module: ", rest)
        res = []
        for p in parts[1:]:
            hdr, _, body = p.partition("\n")
            im = re.search(r"Instance:\s*(\S+)", hdr)
            res.append((im.group(1) if im else "?", body))
        return res

    # Line coverage
    for inst, body in blocks(r"LINE COVERAGE RESULTS"):
        if "Missed Lines" not in body:
            continue
        for m in re.finditer(r"^\s+(\d+):\s+(.*)$", body, re.MULTILINE):
            out["missed_lines"].append((inst, int(m.group(1)), m.group(2).strip()))

    # Toggle coverage
    for inst, body in blocks(r"TOGGLE COVERAGE RESULTS"):
        for m in re.finditer(r"^\s+(\S+)\s+0->1:\s+(\S+)\s*\n\s*\.+\s*1->0:\s+(\S+)", body, re.MULTILINE):
            sig = m.group(1)
            w, m01 = _hex_to_int(m.group(2))
            _, m10 = _hex_to_int(m.group(3))
            if w is None:
                continue
            out["toggles"].append((inst, sig, _missing_bits(w, m01), _missing_bits(w, m10)))

    # Combinational logic coverage
    for inst, body in blocks(r"COMBINATIONAL LOGIC COVERAGE RESULTS"):
        for em in re.finditer(r"^\s+(\d+):\s+(.*?)\n(.*?)(?=\n\s+=====|\Z)", body, re.MULTILINE | re.DOTALL):
            line_no = int(em.group(1))
            expr = re.sub(r"\s+", " ", em.group(2)).strip()
            rest = em.group(3)
            hm = re.search(r"Expression\s+\d+\s+\((\d+)/(\d+)\)", rest)
            hit, total = (int(hm.group(1)), int(hm.group(2))) if hm else (None, None)
            missed_vals = []
            lines = rest.splitlines()
            for i, l in enumerate(lines):
                if l.strip().startswith("=") and "|" in l or re.match(r"^\s*=[^=]+=\s*$", l):
                    star = lines[i + 1] if i + 1 < len(lines) else ""
                    pos = 0
                    for cell in l.split("|"):
                        span = range(pos, pos + len(cell))
                        if any(star[j] == "*" for j in span if j < len(star)):
                            missed_vals.append(cell.strip().strip("="))
                        pos += len(cell) + 1
                    break
            out["logic"].append((inst, line_no, expr, hit, total, missed_vals))
    return out


def coverage_feedback(parsed, max_items=40):
    """Convert uncovered targets into concise, actionable feedback text for the LLM."""
    lines = []
    if parsed["missed_lines"]:
        lines.append("UNCOVERED RTL LINES (never executed):")
        for inst, ln, txt in parsed["missed_lines"][:max_items]:
            lines.append(f"  - {inst}: line {ln}: {txt}")
    tog = sorted([t for t in parsed["toggles"] if t[2] or t[3]], key=lambda t: (t[0].count("."), t[0]))
    if tog:
        lines.append("TOGGLE COVERAGE MISSES (signal bits that never changed in the given direction):")
        for inst, sig, m01, m10 in tog[:max_items]:
            parts = []
            if m01:
                parts.append("never 0->1 on bit(s) " + ",".join(map(str, m01)))
            if m10:
                parts.append("never 1->0 on bit(s) " + ",".join(map(str, m10)))
            lines.append(f"  - {inst}.{sig}: " + "; ".join(parts))
        if len(tog) > max_items:
            lines.append(f"  ... and {len(tog) - max_items} more signals")
    lg = sorted([l for l in parsed["logic"] if l[3] is not None and l[3] < l[4]], key=lambda l: (l[0].count("."), l[0], l[1]))
    if lg:
        lines.append("COMBINATIONAL LOGIC COVERAGE MISSES (expression input combinations never seen; values are the operand bits left-to-right, '-' = don't care):")
        for inst, ln, expr, hit, total, missed in lg[:max_items]:
            mv = (" missed: " + ", ".join(missed)) if missed else ""
            lines.append(f"  - {inst} line {ln}: `{expr}` hit {hit}/{total};{mv}")
        if len(lg) > max_items:
            lines.append(f"  ... and {len(lg) - max_items} more expressions")
    if not lines:
        lines.append("No uncovered targets: line, toggle and logic coverage are all 100%.")
    return "\n".join(lines)


def run_coverage(design, top_module, rtl_files, tb_file, label, show=True):
    """
    One complete coverage measurement (the coverage_exp flow) for a given RTL + testbench.
    Writes everything to ./coverage/<design>/<label>/ and returns a dict with the numbers,
    the parsed uncovered targets and the simulation result.
    """
    outdir = os.path.join(COV_ROOT, design, label)
    os.makedirs(outdir, exist_ok=True)

    with open(tb_file, "r") as f:
        tb_text = f.read()
    tb_top, dut_inst = _parse_tb(tb_text, top_module)

    # Keep a copy of the exact testbench and RTL that were measured.
    shutil.copy(tb_file, os.path.join(outdir, os.path.basename(tb_file)))
    for p in rtl_files:
        shutil.copy(p, os.path.join(outdir, os.path.basename(p)))

    # 1. Wrap the testbench with a waveform dump (no change to the testbench file).
    raw_vcd = os.path.join(outdir, design + "_raw.vcd")
    dump_file = os.path.join(outdir, "cov_dump.v")
    with open(dump_file, "w") as f:
        f.write("// Waveform wrapper added by the coverage flow; the testbench itself is unchanged.\n")
        f.write("module cov_dump;\n  initial begin\n")
        f.write('    $dumpfile("%s");\n' % raw_vcd)
        f.write("    $dumpvars(0, %s);\n  end\nendmodule\n" % tb_top)

    # 2. iverilog + vvp (same commands as coverage_exp, with both roots: testbench + wrapper).
    sim = os.path.join(outdir, design + ".vvp")
    compile_cmd = [IVERILOG, "-Wall", "-Wno-timescale", "-s", tb_top, "-s", "cov_dump", "-o", sim] + list(rtl_files) + [tb_file, dump_file]
    rc, so, se = _run(compile_cmd)
    if rc != 0:
        raise RuntimeError("iverilog failed:\n" + se + so)
    rc, sim_out, sim_err = _run([VVP, sim])
    with open(os.path.join(outdir, "sim_output.txt"), "w") as f:
        f.write(sim_out + sim_err)
    passed = _simulation_passed(sim_out, sim_err) and rc == 0
    n_tests = _count_tests(sim_out)

    # 3. Clean the VCD for Covered 0.7.10.
    vcd = os.path.join(outdir, design + ".vcd")
    _clean_vcd(raw_vcd, vcd)

    # 4. covered score -> CDD
    cdd = os.path.join(outdir, design + ".cdd")
    score_cmd = [COVERED, "score", "-t", top_module, "-i", tb_top + "." + dut_inst]
    for p in rtl_files:
        score_cmd += ["-v", p]
    score_cmd += ["-vcd", vcd, "-o", cdd]
    rc, so, se = _run(score_cmd)
    if rc != 0:
        raise RuntimeError("covered score failed:\n" + so + se)

    # 5. covered report: summary (-d s) and detail (-d v), both per instance (-i) so that every
    #    instance of a reused submodule is measured separately.
    summary = os.path.join(outdir, "coverage_summary.rpt")
    detail = os.path.join(outdir, "coverage_detail.rpt")
    module_summary = os.path.join(outdir, "coverage_summary_by_module.rpt")
    _run([COVERED, "report", "-m", "ltc", "-d", "s", "-i", "-o", summary, cdd])
    _run([COVERED, "report", "-m", "ltc", "-d", "v", "-i", "-o", detail, cdd])
    _run([COVERED, "report", "-m", "ltc", "-d", "s", "-o", module_summary, cdd])

    with open(summary, "r") as f:
        summary_text = f.read()
    with open(detail, "r") as f:
        detail_text = f.read()
    numbers = _parse_summary(summary_text)
    parsed = _parse_detail(detail_text)
    feedback = coverage_feedback(parsed)
    with open(os.path.join(outdir, "uncovered_targets.txt"), "w") as f:
        f.write(feedback)

    result = {
        "design": design, "label": label, "testbench": tb_file, "rtl": list(rtl_files),
        "tests": n_tests, "sim_passed": passed, "numbers": numbers, "parsed": parsed,
        "feedback": feedback, "outdir": outdir, "sim_stdout": sim_out,
        "commands": {
            "iverilog": " ".join(compile_cmd), "vvp": "vvp " + sim,
            "covered_score": " ".join(score_cmd),
            "covered_report": f"covered report -m ltc -d s -i -o {summary} {cdd} ; covered report -m ltc -d v -i -o {detail} {cdd}",
        },
    }
    if show:
        print(f"[{design} / {label}] tests={n_tests} sim_passed={passed}")
        print("   " + format_numbers(numbers))
    return result


def format_numbers(n):
    g = lambda k: f"{n[k]['pct']}% ({n[k]['hit']}/{n[k]['total']})" if k in n else "n/a"
    return f"line {g('line')} | toggle 0->1 {g('toggle01')} | toggle 1->0 {g('toggle10')} | logic {g('logic')}"


def _history_row(label, r):
    n = r["numbers"]
    return {
        "iteration": label, "tests": r["tests"], "sim_passed": r["sim_passed"],
        "line_pct": n.get("line", {}).get("pct"), "line": n.get("line"),
        "toggle01_pct": n.get("toggle01", {}).get("pct"), "toggle01": n.get("toggle01"),
        "toggle10_pct": n.get("toggle10", {}).get("pct"), "toggle10": n.get("toggle10"),
        "logic_pct": n.get("logic", {}).get("pct"), "logic": n.get("logic"),
        "testbench": r["testbench"], "report_dir": r["outdir"],
    }


def print_history(history):
    print(f"{'Iteration':<12}{'# tests':>8}{'Line':>8}{'Tog 0->1':>10}{'Tog 1->0':>10}{'Logic':>8}   sim")
    for h in history:
        print(f"{h['iteration']:<12}{h['tests']:>8}{str(h['line_pct'])+'%':>8}{str(h['toggle01_pct'])+'%':>10}"
              f"{str(h['toggle10_pct'])+'%':>10}{str(h['logic_pct'])+'%':>8}   {'passed' if h['sim_passed'] else 'FAILED'}")


print("Coverage integration ready. iverilog:", IVERILOG, "| covered:", COVERED)

In [ ]:
#@title Coverage-guided testbench refinement (LLM adds targeted tests, existing checks preserved)
# Second feedback loop of the assignment. The RTL is frozen here. The LLM receives the current
# testbench plus the uncovered targets and returns ONLY new test cases, which are inserted at the
# "=== COVERAGE TESTS ===" marker. Every existing check stays untouched by construction.

import difflib

COVERAGE_MARKER = "// === COVERAGE TESTS ==="

ITERATION_LOG = []   # one entry per RTL or TB change, written to iteration_log.json


def log_iteration(design, phase, module_or_tb, what, result):
    ITERATION_LOG.append({"design": design, "phase": phase, "target": module_or_tb, "change": what, "result": result})
    with open("iteration_log.json", "w") as f:
        json.dump(ITERATION_LOG, f, indent=2)


def _strip_fences(text):
    text = re.sub(r"```[a-zA-Z]*\n", "", text)
    return text.replace("```", "")


_STRUCTURAL = re.compile(r"^(module|endmodule|initial|end\b|begin\b|\$finish|\$display|\$dump|reg\b|wire\b|integer\b|"
                         r"task\b|endtask|input\b|output\b|if\s*\(fail_count|else\b|`timescale|" + re.escape("a = 0; b = 0") + ")")


def _extract_new_tests(response):
    """Keep only the new Verilog statements/comments from the LLM reply (no module wrapper, no prose).
    If the model echoed the whole testbench, only the part after the marker is used."""
    txt = _strip_fences(response)
    if COVERAGE_MARKER in txt:
        txt = txt.split(COVERAGE_MARKER, 1)[1]
        txt = re.split(r"\n\s*if\s*\(fail_count|\n\s*\$finish|\nendmodule", txt)[0]
    keep = []
    for line in txt.splitlines():
        s = line.strip()
        if not s or _STRUCTURAL.match(s):
            continue
        if s.startswith("//") or s.endswith(";") or s.endswith("begin") or s == "end":
            keep.append("        " + s)
    return "\n".join(keep) + "\n"


def _splice_tests(tb_text, new_tests):
    idx = tb_text.find(COVERAGE_MARKER)
    if idx < 0:
        raise ValueError("Testbench has no " + COVERAGE_MARKER + " marker")
    eol = tb_text.find("\n", idx) + 1
    return tb_text[:eol] + new_tests + tb_text[eol:]


def _tb_refine_prompt(tb_text, feedback, top_module, n_new):
    return (
        "You are helping improve a Verilog testbench using code-coverage feedback.\n"
        f"The design under test is `{top_module}`. The RTL is correct and FROZEN; do not touch it.\n\n"
        "Below is the CURRENT self-checking testbench. It must be preserved exactly: do NOT rewrite it, "
        "do NOT remove or change any existing check, and do NOT change the task definitions, the DUT "
        "instance name or the final summary.\n\n"
        "```verilog\n" + tb_text + "\n```\n\n"
        "The Covered coverage tool reports these UNCOVERED TARGETS for this testbench:\n\n"
        + feedback + "\n\n"
        f"Add about {n_new} NEW targeted test cases that exercise these uncovered targets "
        "(for example: make the never-toggled bits toggle in both directions, force the missing carry/"
        "overflow conditions, and produce the missing operand combinations in the listed instances). "
        "Reuse the existing `check(...)` task so expected values keep coming from the reference "
        "expression inside the testbench. Each new test should be one statement of the form "
        "`check(a_value, b_value, cin_value);` (or the task's actual argument list), optionally "
        "preceded by a one-line comment saying which uncovered target it is for.\n\n"
        f"Output ONLY the new Verilog statements that will be inserted after the line `{COVERAGE_MARKER}` "
        "inside the initial block. No module header, no explanations, no Markdown fences."
    )


def refine_testbench(design, top_module, rtl_files, tb_file, feedback, iteration, model_type=None, n_new=6):
    """
    Ask the LLM for targeted tests, splice them in, compile against the frozen RTL.
    Returns (new_tb_path, sim_passed, sim_stdout). If the new tests do not compile, the LLM gets one
    retry with the compiler message; if it still fails, the baseline tests are kept and the round is skipped.
    """
    model_type = model_type or os.environ.get("MODEL", "ChatGPT")
    with open(tb_file, "r") as f:
        tb_text = f.read()

    tb_dir = os.path.join(COV_ROOT, design)
    os.makedirs(tb_dir, exist_ok=True)
    new_tb = os.path.join(tb_dir, f"tb_iter_{iteration}.v")
    log_file = os.path.join(tb_dir, f"tb_refine_iter_{iteration}_log.txt")

    conv = Conversation(log_file=log_file)
    conv.add_message("user", _tb_refine_prompt(tb_text, feedback, top_module, n_new))

    for attempt in range(2):
        response = generate_verilog(conv, model_type) or ""
        conv.add_message("assistant", response)
        new_tests = _extract_new_tests(response)
        candidate = _splice_tests(tb_text, new_tests)
        with open(new_tb, "w") as f:
            f.write(candidate)

        tb_top, _ = _parse_tb(candidate, top_module)
        rc, so, se = _run([IVERILOG, "-s", tb_top, "-o", os.path.join(tb_dir, f"tb_iter_{iteration}.vvp")] + list(rtl_files) + [new_tb])
        if rc == 0:
            break
        print(f"  [refine] new tests did not compile (attempt {attempt + 1}):\n{se}")
        conv.add_message("user", "The inserted statements did not compile:\n" + se +
                         "\nOutput ONLY corrected Verilog statements for insertion, nothing else.")
    else:
        print("  [refine] giving up on this round; keeping the previous testbench")
        shutil.copy(tb_file, new_tb)
        return new_tb, True, ""

    added = [l for l in new_tests.splitlines() if l.strip() and not l.strip().startswith("//")]
    print(f"  [refine] {len(added)} new test statements added -> {new_tb}")
    for l in new_tests.splitlines():
        print("     " + l.strip())

    rc, sim_out, sim_err = _run([VVP, os.path.join(tb_dir, f"tb_iter_{iteration}.vvp")])
    passed = rc == 0 and _simulation_passed(sim_out, sim_err)
    return new_tb, passed, sim_out + sim_err


def rome_repair_with_fixed_tb(design, top_module, interface, rtl_files, tb_file, round_no, max_iterations=5, model_type=None):
    """
    Part III Step 4: the stronger testbench exposed an RTL failure. Keep that testbench FIXED and
    return to the ROME RTL-repair loop (verilog_loop) for the top module. Submodules are passed as
    prev_modules exactly as hier_gen does. Returns the path of the repaired top-level module.
    """
    model_type = model_type or os.environ.get("MODEL", "ChatGPT")
    top_file = rtl_files[-1]
    prev = list(rtl_files[:-1])
    with open(top_file, "r") as f:
        current = f.read()
    outdir = os.path.join(".", top_module, f"repair_round_{round_no}")
    prompt = (
        "// The following module FAILED the fixed testbench shown above. Repair it.\n"
        "// Keep the exact module name and interface. Use Verilog-2001 only.\n"
        "// Previously generated submodules may be instantiated but must NOT be redefined.\n\n"
        + current
    )
    verilog_loop(prompt, top_module, tb_file, max_iterations, model_type, outdir, os.path.join(outdir, "log.txt"), prev_modules=prev)
    repaired = os.path.join(outdir, top_module + ".v")
    return repaired


def coverage_loop(design, top_module, interface, rtl_files, baseline_tb, max_rounds=2, n_new=6, model_type=None, max_repair_iterations=5):
    """
    Full Part III driver for one design:
      baseline coverage -> (feedback -> LLM adds tests -> sim -> [RTL repair if it fails] -> coverage) x rounds
    Produces ./coverage/<design>/{baseline,refine_N}/..., tb_iter_N.v, tb_final.v, coverage_history.json.
    """
    rtl_files = list(rtl_files)
    history = []
    tb_dir = os.path.join(COV_ROOT, design)
    os.makedirs(tb_dir, exist_ok=True)

    tb_current = os.path.join(tb_dir, "tb_iter_0.v")
    shutil.copy(baseline_tb, tb_current)

    print("=" * 80 + f"\nBaseline coverage for {design}\n" + "=" * 80)
    r = run_coverage(design, top_module, rtl_files, tb_current, "baseline")
    if not r["sim_passed"]:
        raise RuntimeError("Baseline testbench does not pass: fix the RTL with the ROME loop first.")
    history.append(_history_row("Baseline", r))
    log_iteration(design, "coverage", tb_current, "baseline measurement (no change)", format_numbers(r["numbers"]))

    for rnd in range(1, max_rounds + 1):
        print("\n" + "=" * 80 + f"\nRefinement round {rnd}: uncovered targets fed back to the LLM\n" + "=" * 80)
        print(r["feedback"][:3000] + ("\n[...]" if len(r["feedback"]) > 3000 else ""))
        if "No uncovered targets" in r["feedback"]:
            print("Coverage is already 100%; stopping.")
            break

        new_tb, passed, sim_text = refine_testbench(design, top_module, rtl_files, tb_current, r["feedback"], rnd, model_type, n_new)
        tb_current = new_tb

        if not passed:
            # Step 4: a coverage-guided test exposed an RTL failure. TB is frozen from here on.
            print("\n!!! The stronger testbench FAILED on the current RTL. Freezing the testbench and "
                  "returning to the ROME RTL-repair loop.")
            print(sim_text[-2000:])
            log_iteration(design, "TB", new_tb, f"round {rnd}: added coverage-guided tests", "exposed RTL failure (TB frozen)")
            repaired = rome_repair_with_fixed_tb(design, top_module, interface, rtl_files, new_tb, rnd, max_repair_iterations, model_type)
            rtl_files[-1] = repaired
            r_chk = run_coverage(design, top_module, rtl_files, new_tb, f"refine_{rnd}_after_rtl_repair", show=False)
            log_iteration(design, "RTL", repaired, f"ROME repair with tb_iter_{rnd}.v fixed", "passed" if r_chk["sim_passed"] else "STILL FAILING")
            if not r_chk["sim_passed"]:
                print("RTL repair did not converge; stopping the coverage loop here.")
                history.append(_history_row(f"Refine {rnd}", r_chk))
                break
        else:
            log_iteration(design, "TB", new_tb, f"round {rnd}: added coverage-guided tests", "passed on frozen RTL")

        r = run_coverage(design, top_module, rtl_files, tb_current, f"refine_{rnd}")
        history.append(_history_row(f"Refine {rnd}", r))

    final_tb = os.path.join(tb_dir, "tb_final.v")
    shutil.copy(tb_current, final_tb)
    hist_path = os.path.join(tb_dir, "coverage_history.json")
    with open(hist_path, "w") as f:
        json.dump(history, f, indent=2)

    print("\n" + "=" * 80 + f"\nCoverage history for {design}  (saved to {hist_path}; final TB: {final_tb})\n" + "=" * 80)
    print_history(history)
    return history, final_tb, rtl_files


print("Testbench refinement loop ready.")

In [ ]:
#@title Part III: baseline coverage -> coverage-guided refinement rounds for adder8 (RTL frozen)
adder8_interface = adder_submodules[-1][2]
adder8_history, adder8_final_tb, adder8_final_rtl = coverage_loop(
    design="adder8", top_module="adder8", interface=adder8_interface,
    rtl_files=adder_rtl, baseline_tb="./adder_tb/adder8_tb.v",
    max_rounds=2, n_new=6,
)

In [ ]:
#@title Part III: final stronger testbench, final simulation and commands
print("Final testbench:", adder8_final_tb)
print(open(adder8_final_tb).read())
print("=" * 80)
final_ok = final_sim("adder8", adder8_final_tb, adder8_final_rtl)
print("Final RTL files:", adder8_final_rtl)
last = run_coverage("adder8", "adder8", adder8_final_rtl, adder8_final_tb, "final")
print("\nCommands used for the final run:")
for k, v in last["commands"].items():
    print(f"  {k}: {v}")

In [ ]:
#@title Part I (optional): Covered on the supplied mux64_1 testbench from Part I (no refinement required)
mux_rtl = [f"./{m}/{m}.v" for m in mux_modules]
mux64_cov = run_coverage("mux64_1", "mux64_1", mux_rtl, "./mux64_1tb.v", "baseline")
print(mux64_cov["feedback"][:2500])

In [ ]:
#@title Zip every artifact for the GitHub submission (HW_ROME_Coverage/)
import shutil, os
keep = (mux_modules + adder_modules + ["mux_ppa_runs", "adder_tb", "coverage"]
        + [f"{m}tb.v" for m in mux_modules] + ["iteration_log.json", "run_summary.md"])
stage = "HW_ROME_Coverage"
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
for k in keep:
    if os.path.isdir(k):
        shutil.copytree(k, os.path.join(stage, k), ignore=shutil.ignore_patterns("*_raw.vcd"))
    elif os.path.exists(k):
        shutil.copy(k, stage)
shutil.copy("./coverage/adder8/coverage_history.json", stage)
shutil.make_archive(stage, "zip", ".", stage)
print("Created", stage + ".zip", "->", os.path.getsize(stage + ".zip") // 1024, "KB")
try:
    from google.colab import files
    files.download(stage + ".zip")
except Exception as e:
    print("Download it from the Files panel on the left:", stage + ".zip")